# 🎙️ Transcrição de Vídeos com Whisper
> Baixa vídeos do Google Drive e transcreve com OpenAI Whisper

**Como usar:**
1. `Runtime → Change runtime type → T4 GPU`
2. Adicione os IDs dos vídeos na célula de configuração
3. `Run All`

In [ ]:
# ============================================================
# CONFIGURAÇÃO — edite aqui
# ============================================================
VIDEOS = [
    {
        "id":    "1zEdo8l48ARUGCv179-UlEyWEoedV9vEC",
        "label": "Aula 1 – Protótipo"
    },
    {
        "id":    "1uX1iKBK7ms-12OZnunOpd0-Fw-SLy1Ye",
        "label": "Aula 2 – Protótipo"
    },
]

MODELO = "medium"   # tiny | base | small | medium | large
IDIOMA = "pt"       # pt | en | None (auto-detect)

In [ ]:
# ============================================================
# INSTALAÇÃO
# ============================================================
!pip install -q openai-whisper gdown
!apt-get install -qq ffmpeg
print('✅ Dependências instaladas')

In [ ]:
# ============================================================
# DOWNLOAD DOS VÍDEOS
# ============================================================
import gdown, os

os.makedirs('videos', exist_ok=True)

caminhos = []
for v in VIDEOS:
    destino = f"videos/{v['id']}.mp4"
    if not os.path.exists(destino):
        print(f"⬇️  Baixando: {v['label']}...")
        gdown.download(id=v['id'], output=destino, quiet=False, fuzzy=True)
    else:
        print(f"✅ Já baixado: {v['label']}")
    caminhos.append((v['label'], destino))

print('\n✅ Downloads concluídos')

In [ ]:
# ============================================================
# TRANSCRIÇÃO
# ============================================================
import whisper, json

print(f'🔄 Carregando modelo "{MODELO}"...')
model = whisper.load_model(MODELO)
print('✅ Modelo carregado\n')

resultados = []
for label, path in caminhos:
    print(f'🎙️  Transcrevendo: {label}...')
    opts = {"language": IDIOMA} if IDIOMA else {}
    result = model.transcribe(path, **opts, verbose=False)
    resultados.append({"label": label, "result": result})
    print(f'✅ Concluído: {label}\n')

print('🎉 Todas as transcrições finalizadas!')

In [ ]:
# ============================================================
# VISUALIZAÇÃO FORMATADA
# ============================================================
from IPython.display import display, HTML

def formatar_tempo(seg):
    m, s = divmod(int(seg), 60)
    h, m = divmod(m, 60)
    return f'{h:02d}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

def html_transcricao(label, result):
    linhas = []
    for seg in result['segments']:
        t = formatar_tempo(seg['start'])
        texto = seg['text'].strip()
        linhas.append(
            f'<tr>'
            f'<td style="color:#888;font-family:monospace;white-space:nowrap;padding:2px 12px 2px 0">[{t}]</td>'
            f'<td style="padding:2px 0">{texto}</td>'
            f'</tr>'
        )
    corpo = '\n'.join(linhas)
    return f'''
    <div style="background:#1e1e1e;color:#d4d4d4;padding:16px 20px;border-radius:8px;margin:12px 0;font-size:14px;line-height:1.6">
      <h3 style="color:#4fc3f7;margin:0 0 12px">🎙️ {label}</h3>
      <table style="border-collapse:collapse;width:100%">{corpo}</table>
    </div>'''

for item in resultados:
    display(HTML(html_transcricao(item['label'], item['result'])))

In [ ]:
# ============================================================
# SALVAR ARQUIVOS DE SAÍDA
# ============================================================
import os, json

os.makedirs('transcricoes', exist_ok=True)

for item in resultados:
    slug = item['label'].replace(' ', '_').replace('/', '-')

    # .txt com timestamps
    txt = '\n'.join(
        f"[{formatar_tempo(s['start'])}] {s['text'].strip()}"
        for s in item['result']['segments']
    )
    with open(f"transcricoes/{slug}.txt", 'w') as f:
        f.write(txt)

    # .json completo
    with open(f"transcricoes/{slug}.json", 'w') as f:
        json.dump(item['result'], f, ensure_ascii=False, indent=2)

    print(f"💾 Salvo: transcricoes/{slug}.txt")

print('\n✅ Arquivos salvos em /transcricoes/')